In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import json
import joblib

In [2]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
# from sklearn.
from scipy.stats import loguniform

In [3]:
df = pd.read_parquet('../data/aircraft engine/PM_train.parquet')
df.head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,max,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,187


In [4]:
df['setting2'].value_counts()

setting2
-0.0003    2104
 0.0001    2097
 0.0000    2070
 0.0003    2065
-0.0004    2051
-0.0002    2049
 0.0002    2038
-0.0001    2029
 0.0004    1997
 0.0005    1068
-0.0005     958
 0.0006      71
-0.0006      34
Name: count, dtype: int64

In [5]:
df['s6'].value_counts()
# s6
# 21.61    20225
# 21.60      406
# Name: count, dtype: int64

s6
21.61    20225
21.60      406
Name: count, dtype: int64

In [6]:
df.drop(['setting3', 's1', 's5', 's6', 's10', 's16', 's18', 's19', 'cycle', 'max'], axis=1, inplace=True)

In [7]:
col = df.columns.tolist()

In [8]:
scaler = StandardScaler()
# df = scaler.fit_transform(df)
# df = pd.DataFrame(df, columns=col)
# df.head()

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        20631 non-null  int64  
 1   setting1  20631 non-null  float64
 2   setting2  20631 non-null  float64
 3   s2        20631 non-null  float64
 4   s3        20631 non-null  float64
 5   s4        20631 non-null  float64
 6   s7        20631 non-null  float64
 7   s8        20631 non-null  float64
 8   s9        20631 non-null  float64
 9   s11       20631 non-null  float64
 10  s12       20631 non-null  float64
 11  s13       20631 non-null  float64
 12  s14       20631 non-null  float64
 13  s15       20631 non-null  float64
 14  s17       20631 non-null  int64  
 15  s20       20631 non-null  float64
 16  s21       20631 non-null  float64
 17  RUL       20631 non-null  int64  
dtypes: float64(15), int64(3)
memory usage: 2.8 MB


In [10]:
df.describe()

,id,setting1,setting2,s2,s3,s4,s7,s8,s9,s11,s12,s13,s14,s15,s17,s20,s21,RUL
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000
mean,51.506568,-0.000009,0.000002,642.680934,1590.523119,1408.933782,553.367711,2388.096652,9065.242941,47.541168,521.413470,2388.096152,8143.752722,8.442146,393.210654,38.816271,23.289705,107.807862
std,29.227633,0.002187,0.000293,0.500053,6.131150,9.000605,0.885092,0.070985,22.082880,0.267087,0.737553,0.071919,19.076176,0.037505,1.548763,0.180746,0.108251,68.880990
min,1.000000,-0.008700,-0.000600,641.210000,1571.040000,1382.250000,549.850000,2387.900000,9021.730000,46.850000,518.690000,2387.880000,8099.940000,8.324900,388.000000,38.140000,22.894200,0.000000
25%,26.000000,-0.001500,-0.000200,642.325000,1586.260000,1402.360000,552.810000,2388.050000,9053.100000,47.350000,520.960000,2388.040000,8133.245000,8.414900,392.000000,38.700000,23.221800,51.000000
50%,52.000000,0.000000,0.000000,642.640000,1590.100000,1408.040000,553.440000,2388.090000,9060.660000,47.510000,521.480000,2388.090000,8140.540000,8.438900,393.000000,38.830000,23.297900,103.000000
75%,77.000000,0.001500,0.000300,643.000000,1594.380000,1414.555000,554.010000,2388.140000,9069.420000,47.700000,521.950000,2388.140000,8148.310000,8.465600,394.000000,38.950000,23.366800,155.000000
max,100.000000,0.008700,0.000600,644.530000,1616.910000,1441.490000,556.060000,2388.560000,9244.590000,48.530000,523.380000,2388.560000,8293.720000,8.584800,400.000000,39.430000,23.618400,361.000000


In [11]:
# import seaborn as sns
# sns.histplot(df['RUL'], kde=True, bins=10)
# df['RUL'].value_counts()

In [12]:
x = df.drop('RUL', axis=1)
y = df['RUL']
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.2, random_state=42)
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [13]:
L = Lasso(random_state=42, max_iter=10000, tol=0.001)
R = Ridge(random_state=42, max_iter=10000, tol=0.001)
LR = LinearRegression()
EN = ElasticNet(random_state=42, max_iter=10000, tol=0.001)
RF = RandomForestRegressor(random_state=42)
DT = DecisionTreeRegressor(random_state=42)
GB = GradientBoostingRegressor(random_state=42 , n_iter_no_change=10, validation_fraction=0.2, tol=0.001)

In [14]:
np.logspace(3,5,10, base=5, dtype=int)
# print(9.76562500e-04)
# loguniform(1e-4, 100).rvs(10)
# print(0.001/0.0001)

array([ 125,  178,  255,  365,  522,  747, 1068, 1528, 2185, 3125])

In [15]:
lasso_param = {
    'alpha': np.logspace(-10,2,100, base=2),
    'selection': ['cyclic', 'random'],
    'positive': [True, False]
}

ridge_param = {
    'alpha': np.logspace(-5,4,100, base=2),
    'positive': [True, False]
}

linear_regression_param = {
    'positive': [True, False]
}

elastic_net_param = {
    'alpha': np.logspace(-10,4,100, base=2),
    'l1_ratio': [0.5, 0.3, 0.7],
    'selection': ['cyclic', 'random'],
    'positive': [True, False]
}

random_forest_param = {
    'n_estimators': np.logspace(3,5,10, base=5, dtype=int),
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
    'max_depth':[None, 3, 5, 7, 10],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_features':['sqrt', 'log2', None],
    'bootstrap':[True, False],
}

decision_tree_param = {
    'max_depth':[None, 3, 5, 7, 10],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
    'splitter': ['best', 'random'],
    'max_features':['sqrt', 'log2', None],
}

gradient_boosting_param = {
    'learning_rate':loguniform(1e-4, 100),
    'loss': ['squared_error', 'absolute_error', 'huber', 'quantile'],
    'n_estimators':np.logspace(4, 5, 5, base=5, dtype=int),
    'subsample':[1.0, 0.8, 0.6],
    'criterion':['friedman_mse','squared_error'],
    'max_features':['sqrt', 'log2', None],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_depth':[3, 5, 7, 10],
    'alpha':[0.9,0.8,0.7]
}

In [16]:
grid_lasso = RandomizedSearchCV(estimator=L, param_distributions=lasso_param, cv=5, n_iter=50, random_state=42, scoring='neg_mean_absolute_error', verbose=2)
grid_ridge = RandomizedSearchCV(estimator=R, param_distributions=ridge_param, cv=5, n_iter=50, random_state=42, scoring='neg_mean_absolute_error', verbose=2)
grid_elastic = RandomizedSearchCV(estimator=EN, param_distributions=elastic_net_param, cv=5, n_iter=50, random_state=42, scoring='neg_mean_absolute_error', verbose=2)
grid_linear = GridSearchCV(estimator=LR, param_grid=linear_regression_param, cv=10, scoring='neg_mean_absolute_error', verbose=2)
grid_random = RandomizedSearchCV(estimator=RF, param_distributions=random_forest_param, cv=3, n_iter=10, random_state=42, scoring='neg_mean_absolute_error', verbose=2)
grid_decision = RandomizedSearchCV(estimator=DT, param_distributions=decision_tree_param, cv=5, n_iter=50, random_state=42, scoring='neg_mean_absolute_error', verbose=2)
grid_gradient = RandomizedSearchCV(estimator=GB, param_distributions=gradient_boosting_param, cv=3, n_iter=10, random_state=42, scoring='neg_mean_absolute_error', verbose=2)

In [17]:
grid_lasso.fit(x_train_scaled, y_train)
grid_ridge.fit(x_train_scaled, y_train)
grid_elastic.fit(x_train_scaled, y_train)
grid_linear.fit(x_train_scaled, y_train)
grid_random.fit(x_train_scaled, y_train)
grid_decision.fit(x_train_scaled, y_train)
grid_gradient.fit(x_train_scaled, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END alpha=0.07710831771466045, positive=True, selection=random; total time=   0.0s
[CV] END alpha=0.07710831771466045, positive=True, selection=random; total time=   0.1s
[CV] END alpha=0.07710831771466045, positive=True, selection=random; total time=   0.1s
[CV] END alpha=0.07710831771466045, positive=True, selection=random; total time=   0.1s
[CV] END alpha=0.07710831771466045, positive=True, selection=random; total time=   0.1s
[CV] END alpha=0.34985972139666216, positive=True, selection=cyclic; total time=   0.1s
[CV] END alpha=0.34985972139666216, positive=True, selection=cyclic; total time=   0.1s
[CV] END alpha=0.34985972139666216, positive=True, selection=cyclic; total time=   0.1s
[CV] END alpha=0.34985972139666216, positive=True, selection=cyclic; total time=   0.1s
[CV] END alpha=0.34985972139666216, positive=True, selection=cyclic; total time=   0.1s
[CV] END alpha=0.0019125285102460953, positive=True, selec

RandomizedSearchCV(cv=3,
                   estimator=GradientBoostingRegressor(n_iter_no_change=10,
                                                       random_state=42,
                                                       tol=0.001,
                                                       validation_fraction=0.2),
                   param_distributions={'alpha': [0.9, 0.8, 0.7],
                                        'criterion': ['friedman_mse',
                                                      'squared_error'],
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fb030630e10>,
                                        'loss': ['squared_error',
                                                 'absolute_error', 'huber',
                                                 'quantile'],
                                        'max_depth': [3, 5, 7, 10],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 3, 4, 5],
                                        'min_samples_split': [2, 3, 4, 5, 6, 7,
                                                              9],
                                        'n_estimators': array([ 625,  934, 1397, 2089, 3125]),
                                        'subsample': [1.0, 0.8, 0.6]},
                   random_state=42, scoring='neg_mean_absolute_error',
                   verbose=2)

In [18]:
try:
    path = 'Model.json'
    with open(path, 'r') as file:
        info = json.loads(file.read())
except FileNotFoundError:
    info = {}
    print(info)
    raise ValueError('info is empty')
path

'Model.json'

In [19]:
def save(model):
    y_pred = model.predict(x_test_scaled)
    name = str(model.estimator)
    i = len(info)
    info.update({name+'_'+str(i+1):{'acc':r2_score(y_test,y_pred), 'mae':mean_absolute_error(y_test, y_pred), 'best_p':model.best_params_}})
    with open(path, 'w') as file:
        file.write(json.dumps(info, default=str))

In [20]:
save(grid_lasso)
save(grid_ridge)
save(grid_elastic)
save(grid_linear)
save(grid_random)
save(grid_decision)
save(grid_gradient)

In [23]:
test_df = pd.read_csv('../data/aircraft engine/PM_test.csv')

In [24]:
test_df.head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [25]:
test_df = test_df.drop(['setting3', 's1', 's5', 's6', 's10', 's16', 's18', 's19', 'cycle'], axis=1)

In [26]:
model = GradientBoostingRegressor(
    alpha=0.9,
    criterion="friedman_mse",
    learning_rate=0.0347063453713349,
    loss="squared_error",
    max_depth=10,
    max_features = None,
    min_samples_leaf=3,
    min_samples_split=4,
    n_estimators=625,
    subsample=0.6,
    random_state=42,
    n_iter_no_change=10,
    validation_fraction=0.2,
    tol=0.001
)

In [27]:
model.fit(x_train_scaled, y_train)

GradientBoostingRegressor(learning_rate=0.0347063453713349, max_depth=10,
                          min_samples_leaf=3, min_samples_split=4,
                          n_estimators=625, n_iter_no_change=10,
                          random_state=42, subsample=0.6, tol=0.001,
                          validation_fraction=0.2)

In [30]:
test_df_scaled = scaler.transform(test_df)
y_pred = model.predict(test_df_scaled)

In [31]:
truth_df = pd.read_csv('../data/aircraft engine/PM_truth.csv')

In [33]:
test_df['pred'] = y_pred

In [41]:
test_df[test_df['id']==100]

,id,setting1,setting2,s2,s3,s4,s7,s8,s9,s11,s12,s13,s14,s15,s17,s20,s21,pred
12898,100,0.0014,0.0003,641.65,1591.50,1401.63,554.70,2388.05,9059.87,47.28,522.06,2388.02,8138.54,8.4067,391,39.01,23.3087,177.815113
12899,100,0.0031,0.0001,642.20,1588.99,1402.05,554.05,2387.99,9057.49,47.18,522.14,2388.07,8137.35,8.4291,393,38.97,23.3510,183.799095
12900,100,-0.0000,0.0001,642.27,1587.47,1396.74,554.85,2388.11,9052.23,47.11,522.54,2388.03,8134.63,8.4039,392,39.14,23.3636,191.583737
12901,100,0.0011,0.0001,642.07,1579.17,1401.93,554.05,2388.01,9058.66,47.26,522.34,2388.00,8139.06,8.4057,393,39.04,23.3925,190.230589
12902,100,-0.0011,0.0005,642.01,1589.70,1397.19,553.57,2388.01,9050.85,47.30,522.49,2388.02,8130.34,8.4237,391,38.97,23.4710,187.817960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13091,100,0.0049,0.0000,643.24,1599.45,1415.79,553.41,2388.02,9142.37,47.69,520.69,2388.00,8213.28,8.4715,394,38.65,23.1974,22.763018
13092,100,-0.0011,-0.0001,643.22,1595.69,1422.05,553.22,2388.05,9140.68,47.60,521.05,2388.09,8210.85,8.4512,395,38.57,23.2771,24.951448
13093,100,-0.0006,-0.0003,643.44,1593.15,1406.82,553.04,2388.11,9146.81,47.57,521.18,2388.04,8217.24,8.4569,395,38.62,23.2051,24.871307
13094,100,-0.0038,0.0001,643.26,1594.99,1419.36,553.37,2388.07,9148.85,47.61,521.33,2388.08,8220.48,8.4711,395,38.66,23.2699,21.437330


In [36]:
truth_df

,id,cycle
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91
...,...,...
95,96,137
96,97,82
97,98,59
98,99,117
